# **Semana 3**

## **Introdução**

Nas últimas semanas tratamos do dataset de dados que estava com vários problemas com tipagem errada, dados faltantes, estruturas erradas. Agora para a semana 2, além de termos finalizado o tratamento dos dados, também fizemos a construção de um modelo de previsão utilizando o modelo GBTRegressor. Para esta semana resta-nos apenas criar um modelo de recomendação para a *InsightPlaces*, que foi a última tarefa pendente.

Para esta semana utilizaremos o **KMeans** como nosso modelo de machine learning.

Obs.: Como este projeto se trata de um desafio proposto da Alura, vamos baixar um novo dataset em parquet, simulando o nosso último dataset tratado.

Nesta semana utilizaremos as bibliotecas da semana 1 e 2, juntamente ao Numpy que fornecerá operações matemáticas para nossa função recomendadora.

In [5]:
# import zipfile
# import os
# import requests

# # Caminho do arquivo ZIP
# zip_url = "https://caelum-online-public.s3.amazonaws.com/challenge-spark/semanas-3-e-4.zip"
# zip_path = "./Semanas_zip/semana-2.zip"
# extract_path = "./Dataset"
         
# # Baixar o arquivo ZIP
# response = requests.get(zip_url)
# with open(zip_path, "wb") as f:
#     f.write(response.content)

# # Extrair o conteúdo
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(extract_path)

# # Listar os arquivos extraídos
# os.listdir(extract_path)

In [6]:
from pyspark.sql import SparkSession

# Inicializar o Spark
spark = SparkSession.builder.appName("ProcessamentoParquet").getOrCreate()

# Caminho do arquivo PARQUET
parquet_path = "./Dataset/dataset_ml_parquet"

# Carregar o arquivo JSON em um DataFrame
df = spark.read.parquet(parquet_path)

# Exibir as primeiras linhas
df.show(10)

your 131072x1 screen size is bogus. expect trouble
25/12/26 22:10:10 WARN Utils: Your hostname, Luiz resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/26 22:10:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/26 22:10:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/26 22:10:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+
|                  id|andar|area_util|banheiros|quartos|suites|vaga|         bairro|condominio| iptu|    valor|Zona Central|Zona Norte|Zona Oeste|Zona Sul|Academia|Animais permitidos|Churrasqueira|Condomínio fechado|Elevador|Piscina|Playground|Portaria 24h|Portão eletrônico|Salão de festas|
+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+
|00002dd9-cc74-480...|    2|       35|        1|      1|   0.0| 0.0|   Santo Cristo|     100.0|100.0| 245000.0|           1|

## **Pré-Processamento**

### **VectorAssembler**

E como de começo precisamos aplicar o **VectorAssembler** para estruturar os dados para o modelo. Mas vemos também um pequeno problema, temos colunas `id` e `bairro` que não vão servir para o treinamento do modelo de machine learning.

In [7]:
from pyspark.ml.feature import VectorAssembler

In [8]:
X = df.columns
X.remove('id')
X.remove('bairro')

In [9]:
assembler = VectorAssembler(inputCols=X, outputCol='features')

df_vetorizado = assembler.transform(df)
df_vetorizado = df_vetorizado.select('features')
df_vetorizado.show(10, truncate=False)

25/12/26 22:10:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------------------------------------------------------------------------------------+
|features                                                                                                    |
+------------------------------------------------------------------------------------------------------------+
|[2.0,35.0,1.0,1.0,0.0,0.0,100.0,100.0,245000.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]     |
|(23,[0,1,2,3,5,6,7,8,10,15,17,19,20,22],[1.0,84.0,2.0,2.0,1.0,770.0,105.0,474980.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|(23,[1,2,3,6,7,8,12,14,17],[85.0,2.0,2.0,460.0,661.0,290000.0,1.0,1.0,1.0])                                 |
|(23,[1,2,3,5,6,7,8,11,18,19],[58.0,1.0,2.0,1.0,550.0,550.0,249000.0,1.0,1.0,1.0])                           |
|(23,[1,2,3,4,5,6,8,10],[64.0,2.0,2.0,1.0,1.0,850.0,530000.0,1.0])                                           |
|[0.0,200.0,6.0,4.0,4.0,2.0,2500.0,420.0,2900000.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0]  |
|

### **StandardScaler e PCA**

Como o **KMeans** é baseado em vetores, sempre é importante aplicar o **PCA** para reduzir a dimensionalidade de nossos dados, para focar nas principais componentes dos dados e diminuir o tempo de treinamento do modelo. Agora o **PCA** a dados não normalizados, portanto utilizaremos o **StandardScaler** para normalizar nosso dados e prepará-los para o **PCA**.

In [10]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.feature import PCA

In [11]:
scaler = StandardScaler(inputCol='features', outputCol='scaler_features')

modelo_scaler = scaler.fit(df_vetorizado)
df_scaler = modelo_scaler.transform(df_vetorizado)
df_scaler.show(10,truncate=False)

+------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                    |scaler_features                                                                                                                                                                                                                                                                                                                                                                       |
+---------------------------

Aplicado o **StandardScaler** vamos utilizar o **PCA**.

Antes de escolher qual a dimensão para nosso dados, vamos tomar a dimensão máxima de nosso espaço e escolher quais os principais vetores que melhor dizem sobre os dados, com uma taxa de explicação de 80% para eles.

In [12]:
pca = PCA(k=23, inputCol='scaler_features', outputCol='pca_features')

treino_pca = pca.fit(df_scaler)
treino_pca.explainedVariance

25/12/26 22:10:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/26 22:10:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
25/12/26 22:10:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


DenseVector([0.2655, 0.1721, 0.0913, 0.0544, 0.0522, 0.0466, 0.0443, 0.0416, 0.0347, 0.0272, 0.0244, 0.0201, 0.0192, 0.0176, 0.0155, 0.0139, 0.012, 0.0113, 0.0101, 0.0092, 0.0089, 0.0079, 0.0])

In [13]:
explica_array = treino_pca.explainedVariance.array

soma = 0
stop = 0

for i, valor in enumerate(explica_array):
    if(soma<=0.80):
        soma += explica_array[i]
    elif(stop==0):
        stop=1
        print(f"O íncie até 0.80 é {i-1}\nE seu valor é {soma - explica_array[i]}")

O íncie até 0.80 é 8
E seu valor é 0.7753463656098715


Como da para ver, precisamos apenas de 8 variáveis para explicar 80% do nosso DataFrame

In [14]:
novo_pca = PCA(k=8, inputCol='scaler_features', outputCol='pca_features')

modelo_pca = novo_pca.fit(df_scaler)
df_pca = modelo_pca.transform(df_scaler)

df_final = df_pca.select('pca_features')
df_final = df_final.withColumnRenamed('pca_features', 'features')
df_final.show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                            |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[-6.16512504953394,1.3380985269404198,-1.7052299823815482,-0.533828963054541,0.08903815478584926,-0.3134396180248411,-5.880683742692666,-4.346312930649892]         |
|[-3.2529111812183165,-1.117959183623024,-0.2923895841499691,3.1955388200208477,0.15286306982183612,1.3028930490659758,-0.4369399981012161,0.05205299467369959]      |
|[-1.0611769329629643,-1.668504005869305,-2.3075948278392087,0.10553124125771488,-0.06914386452865356,0.7187181899564254,0.08236222790225037,0.14780594823035864]    

## **KMeans**

Feito o tratamento dos dados, utilizaremos o modelo **Kmeans** para a realização do nosso modelo de recomendação.

Assim como todo modelo, o **Kmeans** também precisa ser avaliado para ver quantos clusteres devemos usar para tomar o melhor modelo possível, por isso optamos por usar a *silhueta* e a *inécia*, métricas essas que vão nos dizer o quão os pontos se encaixaram em cada cluster e o quão longe do centróide ficaram. Outra informação importante é que a *silhueta* varia de 0 a 1, com 1 sendo seu melhor resultado implicando que os dados do cluster formam um esfera em torno do centróide.

In [15]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(featuresCol='features', metricName='silhouette', distanceMeasure='squaredEuclidean')

def calcula_cluster(df, numero_clusters):
    kmeans = KMeans(featuresCol='features', k=numero_clusters, seed=390)
    modelo = kmeans.fit(df)
    inercia = modelo.summary.trainingCost
    silhueta = evaluator.evaluate(modelo.transform(df))
    print("-"*15)
    print(f"Clusteres: {numero_clusters}")
    print(f"Inérica: {inercia}")
    print(f"Silhueta: {silhueta}")

In [16]:
for i in range(2,21):
    calcula_cluster(df_final, i)

---------------
Clusteres: 2
Inérica: 847452.0401333759
Silhueta: 0.45359234219553035
---------------
Clusteres: 3
Inérica: 724760.1600492301
Silhueta: 0.45442321564956556
---------------
Clusteres: 4
Inérica: 655218.1720070234
Silhueta: 0.38059115626686296
---------------
Clusteres: 5
Inérica: 621093.5800852603
Silhueta: 0.26214014836694816
---------------
Clusteres: 6
Inérica: 558005.7374814004
Silhueta: 0.35345545282020874
---------------
Clusteres: 7
Inérica: 523146.17748368403
Silhueta: 0.3829165848796271
---------------
Clusteres: 8
Inérica: 495961.971274356
Silhueta: 0.34719567025318443
---------------
Clusteres: 9
Inérica: 431236.5867810909
Silhueta: 0.4081516480694386
---------------
Clusteres: 10
Inérica: 432156.58253801876
Silhueta: 0.35192490732369525
---------------
Clusteres: 11
Inérica: 351964.682347325
Silhueta: 0.46500518863459506
---------------
Clusteres: 12
Inérica: 315833.77876502735
Silhueta: 0.5131870312181729
---------------
Clusteres: 13
Inérica: 311034.3316676

Apesar do modelo com 17 clusteres apresentar o melhor resultado, pois seu erro (inércia), foi de 226 mil, além de que sua silhueta foi uma dos que mais ficou perto de 1, veremos abaixo que ele não é ideal para a nossa recomendação.

In [30]:
kmeans = KMeans(featuresCol='features', k=17, seed=390)
modelo = kmeans.fit(df_final)

teste = modelo.transform(df_final)
teste.groupBy('prediction').count().orderBy('count', ascending=False).show()


+----------+-----+
|prediction|count|
+----------+-----+
|         0|12323|
|         3| 8737|
|        10| 7281|
|         5| 5978|
|         1| 5949|
|         2| 4954|
|        13| 3825|
|        16| 3777|
|         6| 3389|
|         7| 2991|
|        12| 2115|
|        11| 2093|
|         9| 1960|
|        15|  649|
|         4|  494|
|        14|   27|
|         8|    9|
+----------+-----+



Como podemo ver, o cluster 8 tem apenas 9 imóveis em seu catálogo, o que não vai poder ser um bom modelo sendo que nossa meta é recomendar 10 ímoveis no mínimo. Mas escolhendo 12 veja a seguir que o menor cluster tem 27 imóveis a disposição, além de ter boa *silhueta* e *inercia*. Portanto essa será a nossa quantidade ideal.

In [41]:
kmeans = KMeans(featuresCol='features', k=12, seed=390)
modelo = kmeans.fit(df_final)

teste = modelo.transform(df_final)
teste.groupBy('prediction').count().orderBy('count', ascending=False).show()


+----------+-----+
|prediction|count|
+----------+-----+
|         5|12921|
|        10| 9945|
|         3| 7849|
|         6| 6811|
|         0| 6472|
|         2| 5840|
|        11| 4560|
|         1| 3818|
|         4| 3748|
|         7| 3416|
|         9| 1144|
|         8|   27|
+----------+-----+



### **Pipeline**

Escolhido a quantidade ideal de clusteres, vamos criar um pipeline para nosso modelo para facilitar a reprodutibilidade dele mais tarde.

In [33]:
from pyspark.ml.pipeline import Pipeline

kmeans = KMeans(featuresCol='features', k=12, seed=390)

pipeline = Pipeline(stages=[assembler, modelo_scaler, modelo_pca, kmeans])

In [34]:
modelo_final = pipeline.fit(df)

df_final_clusterizado = modelo_final.transform(df)

df_final_clusterizado.show(5)

+--------------------+-----+---------+---------+-------+------+----+------------+----------+-----+--------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+--------------------+--------------------+--------------------+----------+
|                  id|andar|area_util|banheiros|quartos|suites|vaga|      bairro|condominio| iptu|   valor|Zona Central|Zona Norte|Zona Oeste|Zona Sul|Academia|Animais permitidos|Churrasqueira|Condomínio fechado|Elevador|Piscina|Playground|Portaria 24h|Portão eletrônico|Salão de festas|            features|     scaler_features|        pca_features|prediction|
+--------------------+-----+---------+---------+-------+------+----+------------+----------+-----+--------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+-----

## **Recomendação**

Criaremos por fim uma função recomendadora baseada nos 10 imóveis mais próximos cluster a cluster.

Segue o que iremos utilizar para fazer tudo isso.

In [35]:
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.functions import udf
from pyspark.sql.functions import col

import numpy as np

In [36]:
def vetores_cluster(df, numero_cluster):
    cluster = df.filter(df['prediction'] == numero_cluster)

    partes_cluster = cluster.collect()
    vetores_cluster = []

    for i in range(cluster.count()):
        partes = partes_cluster[i]
        id_cluster = [parte for parte in partes][0]
        feature_cluster = [parte for parte in partes][-2]

        vetores_cluster.append((id_cluster, feature_cluster))


    return vetores_cluster

Para conseguirmos aplicar a recomendação dentro do DataFrame do Spark, faremos o seguinte procedimento.

In [56]:
clusteres = []

for numero_cluster in range(0, 12):
    clusteres.append(vetores_cluster(df_final_clusterizado, numero_cluster))

In [54]:
@udf(returnType=ArrayType(StringType()))
def calcula_10_proximos(id_imovel, vetor_imovel, cluster_imovel):
    cluster_utilizado = clusteres[cluster_imovel]

    distancias = []

    for i in range(len(cluster_utilizado)):
        id, vetor = cluster_utilizado[i]

        if id != id_imovel:
           distancias.append((id, np.linalg.norm(vetor - vetor_imovel)))

    distancias = sorted(distancias, key=lambda x: x[1])
    
    distancias_10 = []
    for i in range(10):
        ids, valor = distancias[i]
        distancias_10.append(ids) 

    return distancias_10

In [57]:
df_finalizado = df_final_clusterizado.select('*')

df_finalizado = df_finalizado.withColumn('10 Próximos', calcula_10_proximos(col('id'), col('pca_features'), col('prediction')))
df_finalizado = df_finalizado.drop(*['features', 'scaler_features', 'pca_features', 'prediction'])

df_finalizado.show(10)

+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+--------------------+
|                  id|andar|area_util|banheiros|quartos|suites|vaga|         bairro|condominio| iptu|    valor|Zona Central|Zona Norte|Zona Oeste|Zona Sul|Academia|Animais permitidos|Churrasqueira|Condomínio fechado|Elevador|Piscina|Playground|Portaria 24h|Portão eletrônico|Salão de festas|         10 Próximos|
+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+--------------------+
|00002dd9-cc74-480...|    2|       35|        1|      1|   0.

Com isto, finalizamos a semana fazendo nosso pré-modelo de recomendação, colocaremos tudo agora em um arquivo python para finalizar e prever para quantas imóveis próximos quisermos

## Conclusão

Após processarmos os dados e escolhermos a quantidade ideal de clusteres, nossa função recomendadora finalmente ficou pronta. Com tudo isso seremos capaz de recomendar imóveis com base em um escolhido, já que o modelo é baseado em clusteres, podendo recomendar para ímoveis com características muito próximas, sendo excelente para visão de negócios já que não precisaremos nos preocupar em recomendar apenas imóveis de mesmos bairros. Com isso, quando um cliente escolher um imóvel, ele considerará outros imóveis de mesmo cluster, juntando parecidos, mas sempre escolhendo aquele que mais se aproxima (assemelha) ao que o cliente escolheu.

Dessa forma conseguimos resolver o problema de imóveis sendo recomendados apenas por seus bairros, mas sim possuindo características semelhantes uns aos outros.